# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [1]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display
from tqdm.auto import tqdm

from _bootstrap import ROOT
from utils import BaselineRoute, BaselineRouteGenerator, JeepneySystem, JeepneyRouteEnv, SystemicFitnessEvaluator, calculate_route_fitness, train_route_agent

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


,route_id,anchors,area_m2,length_m,attempt
0,B401,"[1018, 1406, 908, 3692]",69703128.0,61587.0,1
1,B402,"[1071, 1240, 101, 1685]",6845578.0,26414.0,1
2,B403,"[1681, 129, 1090, 3124]",5975720.0,18489.0,1
3,B404,"[36, 1795, 1351, 771]",5050245.0,24525.0,1
4,B405,"[557, 217, 473, 1090]",5466196.0,17740.0,1
5,B406,"[2828, 34, 1685, 1240]",29434398.0,42589.0,1
6,B407,"[795, 1098, 641, 3516]",7122488.0,46172.0,1
7,B408,"[1680, 703, 1003, 88]",32894460.0,35824.0,2
8,B409,"[177, 30, 197, 381]",24729861.0,43637.0,1
9,B410,"[3197, 997, 4, 698]",19265728.0,82480.0,1


## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)

def print_observation(obs: dict[str, np.ndarray]) -> None:
    print('shape:', obs['shape'])
    print('history:', obs['history'])
    print('topology:', obs['topology'])
    print('demand:', obs['demand'])
    print('global:', obs['global'])
    print('candidates:')
    for candidate_index, row in enumerate(obs['candidates']):
        print(f'  {candidate_index}: {row}')
    print('mask:', obs['action_mask'])

obs, info = env.reset()
print('reset state vector length:', info['state_vector'].shape[0])
print_observation(obs)

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('turn angle rad:', info['turn_angle_rad'])
    print('sinuosity index:', info['sinuosity_index'])
    print('distance to origin m:', info['distance_to_origin_m'])
    print('bearing to origin rad:', info['bearing_to_origin_rad'])
    print('route area m2:', info['route_area_m2'])
    print('state vector:', info['state_vector'])
    print_observation(obs)
    if terminated or truncated:
        break


reset state vector length: 91
shape: [0. 0. 1. 0. 0. 0. 0.]
history: [0. 0. 0. 0. 0. 0. 0. 0.]
topology: [0.5   0.5   0.    0.5   0.556]
demand: [0.226 0.357 0.511 0.285]
global: [0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.5   0.5   0.    0.5   0.556 0.226 0.357 0.511 0.285]
candidates:
  0: [-0.389 -0.921  0.011  0.667  0.     0.511  0.285  0.     1.     0.   ]
  1: [-0.295  0.956  0.017  0.5    0.     0.28   0.053  0.     0.     0.   ]
  2: [0.618 0.786 0.009 0.5   0.    0.28  0.053 0.    0.    0.   ]
  3: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  4: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  5: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
mask: [1. 1. 1. 0. 0. 0. 0.]
step 1: action=1, reward=0.250, terminated=False, truncated=False
turn angle rad: 0.0
sinuosity index: 1.0
distance to origin m: 117.46934991757044
bearing to origin rad: 3.141592653589793
route area m2: 0.0
state vector: [ 0.002  0.    -1.     0.002  0.002  0.     0.     0.     0.     0.
  0.     1.   

In [3]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

def build_route_encoding(route_id: str) -> str:
    return f"route_id: {route_id}"

def interpret_embeddings(route: BaselineRoute) -> str:
    """Generate human-readable interpretations of route geometry."""
    route_path = route.path_node_ids
    if not route_path or len(route_path) < 2:
        return "Route too short to analyze."
    
    lines = []
    coords = route.path_latlon
    
    area = route.polygon_area_m2
    length = route.path_length_m
    area_to_length = area / max(length, 1.0)
    
    if area_to_length > 1000:
        lines.append("shape: Compact, efficient use of space")
    elif area_to_length > 200:
        lines.append("shape: Moderate coverage relative to distance")
    else:
        lines.append("shape: Linear route with extended length")
    
    node_count = len(route_path)
    if node_count > 50:
        lines.append("topology: High node density, complex route structure")
    elif node_count > 25:
        lines.append("topology: Moderate complexity with multiple waypoints")
    else:
        lines.append("topology: Simple, direct route path")
    
    if area > 50_000_000:
        lines.append("demand: Large service area, potentially high demand coverage")
    elif area > 10_000_000:
        lines.append("demand: Medium service area with moderate demand potential")
    else:
        lines.append("demand: Compact service area with focused demand")
    
    if len(coords) >= 2:
        import math
        start = coords[0]
        end = coords[-1]
        straight_dist = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2) * 111000
        actual_path = length
        sinuosity = actual_path / max(straight_dist, 1.0)
        
        if sinuosity > 1.5:
            lines.append("history: Winding route with many turns")
        elif sinuosity > 1.1:
            lines.append("history: Moderate turning pattern")
        else:
            lines.append("history: Direct path with minimal deviations")
    
    return chr(10).join(lines) if lines else "No embedding data available."

def format_route_fitness(result) -> str:
    return (
        f"standalone fitness: reward={result.reward:.3f} | avg_gtc={result.average_gtc:.3f} | "
        f"passenger_std={result.passenger_gtc_std:.3f}"
    )

def format_systemic_fitness(result) -> str:
    return (
        f"systemic fitness: avg_gtc={result.average_gtc:.3f} ± {result.std_gtc:.3f} | "
        f"avg_passenger_std={result.average_passenger_gtc_std:.3f} ± {result.std_passenger_gtc_std:.3f}"
    )

NOTE_SYSTEMIC_EVAL_TEST_MEAN = 3  # Lightweight note-time sample; increase later for final analysis.
NOTE_SYSTEMIC_EVAL_TEST_STD = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD = 0
FULL_SYSTEMIC_EVAL_TEST_MEAN = 10  # Temporary placeholder; increase later for final sensitivity analysis.
FULL_SYSTEMIC_EVAL_TEST_STD = 5
FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 2
FULL_SYSTEMIC_BACKGROUND_ROUTE_STD = 1

note_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=NOTE_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=NOTE_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)
full_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=FULL_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=FULL_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=FULL_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)

standalone_fitness_cache: dict[tuple[int, ...], object] = {}
systemic_fitness_cache: dict[tuple[int, ...], object] = {}

def route_signature(route) -> tuple[int, ...]:
    return tuple(int(node_id) for node_id in route.path_node_ids)

def get_standalone_fitness(route):
    key = route_signature(route)
    cached = standalone_fitness_cache.get(key)
    if cached is None:
        cached = calculate_route_fitness(
            route.path_node_ids,
            passenger_map=generator.passenger_map,
            drive_graph_raw=generator.drive_graph_raw,
            drive_graph_proj=generator.drive_graph_proj,
            seed=random_seed,
            batch_size=5,
        )
        standalone_fitness_cache[key] = cached
    return cached

def get_systemic_fitness(route, *, use_full: bool = False):
    key = route_signature(route)
    cache = systemic_fitness_cache
    cached = cache.get((key, use_full))
    if cached is None:
        evaluator = full_systemic_evaluator if use_full else note_systemic_evaluator
        cached = evaluator.evaluate(route, seed=random_seed)
        cache[(key, use_full)] = cached
    return cached

def build_route_nodes(summary_df: pd.DataFrame, routes: list) -> dict[str, dict[str, str]]:
    notes: dict[str, dict[str, str]] = {}
    route_by_id = {r.route_id: r for r in routes}
    total = len(summary_df)
    if total == 0:
        return notes
    
    for row in tqdm(summary_df.itertuples(index=False), total=total, desc="Building route notes"):
        route = route_by_id.get(row.route_id)
        interpretation = interpret_embeddings(route) if route else "Route data unavailable."
        if route is not None:
            standalone = get_standalone_fitness(route)
            systemic = get_systemic_fitness(route)
            interpretation = (
                f"{format_route_fitness(standalone)}\n"
                f"{format_systemic_fitness(systemic)}\n\n"
                f"{interpretation}"
            )
        notes[row.route_id] = {
            "encoding": build_route_encoding(row.route_id),
            "interpretation": interpretation,
        }
    return notes

route_notes = build_route_nodes(summary, routes)


Building route notes:   0%|          | 0/20 [00:00<?, ?it/s]

## 3. 3-Layer Graph Reward Formulation
Connect the generated physical shape to the 3-layer behavioral passenger graph to calculate the fitness score (Reward).

In [4]:
comparison_generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=1,
)
comparison_routes = comparison_generator.generate_routes(2, route_prefix='CMP', seed=1)
good_shape = comparison_routes[0].path_node_ids
bad_shape = comparison_routes[1].path_node_ids

good_result = calculate_route_fitness(
    good_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)
bad_result = calculate_route_fitness(
    bad_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)

comparison = pd.DataFrame(
    [
        {"route_shape": "good", "reward": round(good_result.reward, 3), "avg_gtc": round(good_result.average_gtc, 3), "unserved": good_result.unserved_passenger_count},
        {"route_shape": "bad", "reward": round(bad_result.reward, 3), "avg_gtc": round(bad_result.average_gtc, 3), "unserved": bad_result.unserved_passenger_count},
    ]
)
print(f"Good route reward: {good_result.reward:.3f}")
print(f"Bad route reward: {bad_result.reward:.3f}")
comparison


Good route reward: -174.980
Bad route reward: -176.625


,route_shape,reward,avg_gtc,unserved
0,good,-174.980,174.980,0
1,bad,-176.625,176.625,0


## 4. Systemic Synergy Evaluation Utility
Build a statistical evaluation utility that tests a candidate route across multiple background networks to measure true system synergy.

In [5]:
systemic_result = full_systemic_evaluator.evaluate(routes[0], seed=random_seed)
print(f"Systemic average GTC: {systemic_result.average_gtc:.3f}")
print(f"Systemic GTC std dev: {systemic_result.std_gtc:.3f}")
print(f"Passenger GTC std avg: {systemic_result.average_passenger_gtc_std:.3f}")
print(f"Tests used: {systemic_result.n_tests}")


Systemic average GTC: 6152.394
Systemic GTC std dev: 1312.061
Passenger GTC std avg: 14689.644
Tests used: 14


## For Evaluation: Export HTML Tester

In [6]:
html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_baseline_routes\B4_route_explorer.html


## 5. Standalone Rejection Sampling Setup
This cell rebuilds the route generator and systemic evaluator so part 2 can run without any earlier notebook state.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import BaselineRouteGenerator, SystemicFitnessEvaluator

SAMPLE_SIZE = 5000
GOOD_PERCENTILE = 95.0
ELITE_PERCENTILE = 99.0
RISK_AVERSION_K = 1.0
ROBUSTNESS_CV_CEILING = 0.20
SCREENING_TESTS = 1
VALIDATION_TESTS = 10
SCREENING_BACKGROUND_ROUTE_MEAN = 1
SCREENING_BACKGROUND_ROUTE_STD = 0
VALIDATION_BACKGROUND_ROUTE_MEAN = 2
VALIDATION_BACKGROUND_ROUTE_STD = 1
RANDOM_SEED = 20260422
OUTPUT_YAML = REPO_ROOT / "configs" / "B4_rejection_sampling.yaml"

sampler = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=RANDOM_SEED,
)
screening_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=SCREENING_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=SCREENING_BACKGROUND_ROUTE_MEAN,
    background_route_std=SCREENING_BACKGROUND_ROUTE_STD,
    seed=RANDOM_SEED,
)
validation_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=VALIDATION_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=VALIDATION_BACKGROUND_ROUTE_MEAN,
    background_route_std=VALIDATION_BACKGROUND_ROUTE_STD,
    seed=RANDOM_SEED,
)
route_seed_rng = np.random.default_rng(RANDOM_SEED)
validation_seed_rng = np.random.default_rng(RANDOM_SEED + 1)

def risk_adjusted_score(result) -> float:
    return float(-(float(result.average_gtc) + (RISK_AVERSION_K * float(result.std_gtc))))

def coefficient_of_variation(result) -> float:
    mean_gtc = float(result.average_gtc)
    if mean_gtc <= 0:
        return float("inf")
    return float(result.std_gtc / mean_gtc)

print(f"Standalone setup ready: {SAMPLE_SIZE:,} candidates, P{GOOD_PERCENTILE:.0f}/P{ELITE_PERCENTILE:.0f} thresholds")


Standalone setup ready: 5,000 candidates, P95/P99 thresholds


### 5.1 Sample and screen candidate routes
This cell generates 10,000 candidates with a progress bar and scores each one against the systemic evaluator.


In [2]:
screening_rows = []
candidate_by_id = {}

for index in tqdm(range(SAMPLE_SIZE), desc="Sampling candidate routes", unit="route"):
    while True:
        route_seed = int(route_seed_rng.integers(0, np.iinfo(np.int32).max))
        try:
            route = sampler.generate_route(route_id=f"B4Q{index + 1:05d}", seed=route_seed)
            break
        except ValueError:
            continue

    candidate_by_id[route.route_id] = route
    result = screening_evaluator.evaluate(route, n_tests=SCREENING_TESTS, seed=route_seed)
    screening_rows.append(
        {
            "route_id": route.route_id,
            "average_gtc": float(result.average_gtc),
            "std_gtc": float(result.std_gtc),
            "risk_adjusted_score": risk_adjusted_score(result),
            "reward": float(result.reward),
            "average_passenger_gtc_std": float(result.average_passenger_gtc_std),
            "n_tests": int(result.n_tests),
            "area_m2": float(route.polygon_area_m2),
            "length_m": float(route.path_length_m),
            "attempt": int(route.attempt_index),
            "coefficient_of_variation": coefficient_of_variation(result),
        }
    )

screening_scores = pd.DataFrame(screening_rows).sort_values("risk_adjusted_score", ascending=False).reset_index(drop=True)
display(screening_scores.head(10))


Sampling candidate routes:   0%|          | 0/5000 [00:00<?, ?route/s]

KeyboardInterrupt: 

### 5.2 Filter for robust routes
This cell applies percentile cutoffs, the mean-plus-two-sigma check, and the coefficient-of-variation ceiling.


In [3]:
score_values = screening_scores["risk_adjusted_score"].to_numpy(dtype=float)
mean_gtc = float(screening_scores["average_gtc"].mean())
std_gtc = float(screening_scores["std_gtc"].mean())
mean_risk_adjusted_score = float(score_values.mean())
std_risk_adjusted_score = float(score_values.std(ddof=0))
good_threshold = float(np.percentile(score_values, GOOD_PERCENTILE))
elite_threshold = float(np.percentile(score_values, ELITE_PERCENTILE))
sigma_threshold = float(mean_risk_adjusted_score + 2.0 * std_risk_adjusted_score)
robustness_limit = float(ROBUSTNESS_CV_CEILING)

good_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= good_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
elite_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= elite_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
sigma_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= sigma_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
good_routes = [candidate_by_id[row.route_id] for row in good_scores.itertuples(index=False)]

print(f"Accepted {len(good_routes)} routes at P{GOOD_PERCENTILE:.0f}")
print(f"Risk-adjusted threshold: {good_threshold:.3f}")
print(f"Mean + 2σ threshold: {sigma_threshold:.3f}")
print(f"Robustness ceiling (CV): {robustness_limit:.2f}")
display(good_scores.head(10))


Accepted 249 routes at P95
Risk-adjusted threshold: -3720.032
Mean + 2σ threshold: -3104.456
Robustness ceiling (CV): 0.20


,route_id,average_gtc,std_gtc,risk_adjusted_score,reward,average_passenger_gtc_std,n_tests,area_m2,length_m,attempt,coefficient_of_variation
0,B4Q01373,584.016285,0.0,-584.016285,-584.016285,1856.534064,1,4.165664e+06,12952.829455,3,0.0
1,B4Q00947,1437.144167,0.0,-1437.144167,-1437.144167,5647.905898,1,3.320708e+07,38808.547887,1,0.0
2,B4Q03418,1456.575984,0.0,-1456.575984,-1456.575984,4733.531899,1,1.232111e+07,41967.340706,1,0.0
3,B4Q03413,1613.325502,0.0,-1613.325502,-1613.325502,6390.723852,1,6.034260e+07,67157.961309,1,0.0
4,B4Q01339,1788.059585,0.0,-1788.059585,-1788.059585,6744.595320,1,5.933165e+07,66737.776649,1,0.0
5,B4Q04581,1992.667177,0.0,-1992.667177,-1992.667177,7685.344781,1,1.173095e+08,69086.150437,1,0.0
6,B4Q00276,2139.862964,0.0,-2139.862964,-2139.862964,6701.586275,1,4.148586e+07,67972.672295,1,0.0
7,B4Q00097,2189.740142,0.0,-2189.740142,-2189.740142,7765.408860,1,1.468155e+08,70868.537385,1,0.0
8,B4Q02904,2255.886026,0.0,-2255.886026,-2255.886026,7389.822856,1,7.076008e+06,35271.803179,1,0.0
9,B4Q00479,2264.065078,0.0,-2264.065078,-2264.065078,8418.232429,1,1.378964e+07,70194.717999,1,0.0


### 5.3 Validate and export
This cell re-evaluates the accepted routes with a broader systemic pass, then saves the run settings to YAML.


In [4]:
validation_rows = []
for route in tqdm(good_routes, desc="Validating accepted routes", unit="route"):
    validation_seed = int(validation_seed_rng.integers(0, np.iinfo(np.int32).max))
    validation = validation_evaluator.evaluate(route, n_tests=VALIDATION_TESTS, seed=validation_seed)
    validation_rows.append(
        {
            "route_id": route.route_id,
            "average_gtc": float(validation.average_gtc),
            "std_gtc": float(validation.std_gtc),
            "risk_adjusted_score": risk_adjusted_score(validation),
            "reward": float(validation.reward),
            "average_passenger_gtc_std": float(validation.average_passenger_gtc_std),
            "n_tests": int(validation.n_tests),
            "coefficient_of_variation": coefficient_of_variation(validation),
        }
    )

validation_scores = pd.DataFrame(
    validation_rows,
    columns=[
        "route_id",
        "average_gtc",
        "std_gtc",
        "risk_adjusted_score",
        "reward",
        "average_passenger_gtc_std",
        "n_tests",
        "coefficient_of_variation",
    ],
).sort_values("risk_adjusted_score", ascending=False).reset_index(drop=True)

yaml_payload = {
    "experiment": "B4_rejection_sampling",
    "higher_is_better": True,
    "sample_size": SAMPLE_SIZE,
    "random_seed": RANDOM_SEED,
    "risk_aversion_k": RISK_AVERSION_K,
    "robustness_cv_ceiling": ROBUSTNESS_CV_CEILING,
    "route_generator": {
        "min_area_m2": 2_000_000,
        "anchor_pool_size": 96,
        "max_attempts": 500,
        "route_prefix": "B4Q",
    },
    "screening": {
        "good_percentile": GOOD_PERCENTILE,
        "elite_percentile": ELITE_PERCENTILE,
        "tests": SCREENING_TESTS,
        "background_route_mean": SCREENING_BACKGROUND_ROUTE_MEAN,
        "background_route_std": SCREENING_BACKGROUND_ROUTE_STD,
        "percentile_threshold": good_threshold,
        "elite_threshold": elite_threshold,
        "mean_plus_2sigma_threshold": sigma_threshold,
        "robustness_ceiling": robustness_limit,
    },
    "validation": {
        "tests": VALIDATION_TESTS,
        "background_route_mean": VALIDATION_BACKGROUND_ROUTE_MEAN,
        "background_route_std": VALIDATION_BACKGROUND_ROUTE_STD,
    },
    "summary": {
        "mean_gtc": mean_gtc,
        "std_gtc": std_gtc,
        "mean_risk_adjusted_score": mean_risk_adjusted_score,
        "std_risk_adjusted_score": std_risk_adjusted_score,
        "good_route_count": int(len(good_scores)),
        "elite_route_count": int(len(elite_scores)),
        "sigma_route_count": int(len(sigma_scores)),
        "best_route_id": str(screening_scores.iloc[0]["route_id"]),
        "best_risk_adjusted_score": float(screening_scores.iloc[0]["risk_adjusted_score"]),
        "best_average_gtc": float(screening_scores.iloc[0]["average_gtc"]),
        "best_std_gtc": float(screening_scores.iloc[0]["std_gtc"]),
        "validation_mean_gtc": float(validation_scores["average_gtc"].mean()) if not validation_scores.empty else None,
        "validation_std_gtc": float(validation_scores["average_gtc"].std(ddof=0)) if len(validation_scores) > 1 else 0.0 if len(validation_scores) == 1 else None,
        "validation_mean_risk_adjusted_score": float(validation_scores["risk_adjusted_score"].mean()) if not validation_scores.empty else None,
        "validation_std_risk_adjusted_score": float(validation_scores["risk_adjusted_score"].std(ddof=0)) if len(validation_scores) > 1 else 0.0 if len(validation_scores) == 1 else None,
        "validation_best_route_id": str(validation_scores.iloc[0]["route_id"]) if not validation_scores.empty else None,
    },
}
OUTPUT_YAML.write_text(yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=False), encoding="utf-8")

print(f"Validation routes: {len(validation_scores)}")
print(f"YAML saved to: {OUTPUT_YAML}")
display(validation_scores.head(10))


Validating accepted routes:   0%|          | 0/249 [00:00<?, ?route/s]

Validation routes: 249
YAML saved to: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\configs\B4_rejection_sampling.yaml


,route_id,average_gtc,std_gtc,risk_adjusted_score,reward,average_passenger_gtc_std,n_tests,coefficient_of_variation
0,B4Q00856,5145.015613,595.755897,-5740.771511,-5740.771511,12829.769288,10,0.115793
1,B4Q01378,5093.571467,799.238544,-5892.810011,-5892.810011,13118.263282,10,0.156911
2,B4Q02357,5291.279273,723.088061,-6014.367334,-6014.367334,13780.367056,10,0.136657
3,B4Q02260,5064.054653,958.749562,-6022.804214,-6022.804214,13461.513261,10,0.189324
4,B4Q04971,5238.917559,785.871973,-6024.789532,-6024.789532,13298.617924,10,0.150007
5,B4Q04052,5226.821644,802.088618,-6028.910262,-6028.910262,13274.567208,10,0.153456
6,B4Q00790,5232.022652,826.592274,-6058.614926,-6058.614926,13406.269579,10,0.157987
7,B4Q00996,4956.322340,1105.929948,-6062.252287,-6062.252287,13261.620795,10,0.223135
8,B4Q03400,5417.665172,661.830636,-6079.495808,-6079.495808,13627.323008,10,0.122162
9,B4Q03563,4970.215794,1149.257633,-6119.473426,-6119.473426,12675.074836,10,0.231229


### Optimization log

- Added Cython helpers for anchor ordering, cost reduction, nearest-node lookup, and shortest-path edge reconstruction.
- Added Cython build support and kept a pure-Python fallback for parity.
- Removed eager graph construction in validation and reused cached node geometry plus per-edge weights in the travel graph.
- Cached passenger nearest-node and shortest-path resolution so path prep does not repeat the same lookup twice.
- Reused the systemic validation process pool across route checks, but kept small batches on the single-process path so multiprocessing overhead does not dominate.
- Kept part 3 append-only and resumable, with progress bars and JSONL-first persistence.


## 6. Batch Good Route Library
This section keeps the old 5000-sample workflow: generate a batch, take the top 5% by the part 2 score, validate those survivors, and snapshot the library.


### 6.1 Standalone setup
The next cell reloads the part 2 thresholds from disk, recreates the generator and systemic evaluators, and prepares the batch snapshot path so this part can run on its own.


In [4]:
from pathlib import Path
import json
import secrets
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import BaselineRouteGenerator, SystemicFitnessEvaluator

PART2_CONFIG = REPO_ROOT / 'configs' / 'B4_rejection_sampling.yaml'
if not PART2_CONFIG.exists():
    raise FileNotFoundError(f'Part 2 config not found: {PART2_CONFIG}')

PART2_SETTINGS = yaml.safe_load(PART2_CONFIG.read_text(encoding='utf-8')) or {}
GENERATOR_SETTINGS = PART2_SETTINGS.get('route_generator', {})
SCREENING_SETTINGS = PART2_SETTINGS.get('screening', {})
VALIDATION_SETTINGS = PART2_SETTINGS.get('validation', {})

RISK_AVERSION_K = float(PART2_SETTINGS.get('risk_aversion_k', 1.0))
ROBUSTNESS_CV_CEILING = float(SCREENING_SETTINGS.get('robustness_ceiling', PART2_SETTINGS.get('robustness_cv_ceiling', 0.20)))
SCREENING_SCORE_THRESHOLD = SCREENING_SETTINGS.get('percentile_threshold')
if SCREENING_SCORE_THRESHOLD is None:
    raise ValueError(f'Part 2 config at {PART2_CONFIG} is missing screening.percentile_threshold')
SCREENING_SCORE_THRESHOLD = float(SCREENING_SCORE_THRESHOLD)
SCREENING_ELITE_THRESHOLD = float(SCREENING_SETTINGS.get('elite_threshold', SCREENING_SCORE_THRESHOLD))
SCREENING_SIGMA_THRESHOLD = float(SCREENING_SETTINGS.get('mean_plus_2sigma_threshold', SCREENING_SCORE_THRESHOLD))
GOOD_PERCENTILE = float(SCREENING_SETTINGS.get('good_percentile', 95.0))
ELITE_PERCENTILE = float(SCREENING_SETTINGS.get('elite_percentile', 99.0))
SCREENING_TESTS = int(SCREENING_SETTINGS.get('tests', 1))
VALIDATION_TESTS = int(VALIDATION_SETTINGS.get('tests', 10))
SCREENING_BACKGROUND_ROUTE_MEAN = int(SCREENING_SETTINGS.get('background_route_mean', 1))
SCREENING_BACKGROUND_ROUTE_STD = int(SCREENING_SETTINGS.get('background_route_std', 0))
VALIDATION_BACKGROUND_ROUTE_MEAN = int(VALIDATION_SETTINGS.get('background_route_mean', 2))
VALIDATION_BACKGROUND_ROUTE_STD = int(VALIDATION_SETTINGS.get('background_route_std', 1))
LIBRARY_REFRESH_EVERY = max(int(PART2_SETTINGS.get('library_refresh_every', 25)), 1)
LIBRARY_ROUTE_PREFIX = 'B4L'

RUN_SEED = int(secrets.randbits(32))
OUTPUT_DIR = REPO_ROOT / 'results' / 'B4_good_routes'
OUTPUT_JSONL = OUTPUT_DIR / 'good_routes.jsonl'
OUTPUT_CSV = OUTPUT_DIR / 'good_routes.csv'
OUTPUT_YAML = REPO_ROOT / 'configs' / 'B4_good_route_library.yaml'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_YAML.parent.mkdir(parents=True, exist_ok=True)

sampler = BaselineRouteGenerator(
    min_area_m2=int(GENERATOR_SETTINGS.get('min_area_m2', 2_000_000)),
    anchor_pool_size=int(GENERATOR_SETTINGS.get('anchor_pool_size', 96)),
    max_attempts=int(GENERATOR_SETTINGS.get('max_attempts', 500)),
    seed=RUN_SEED,
)
screening_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=SCREENING_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=SCREENING_BACKGROUND_ROUTE_MEAN,
    background_route_std=SCREENING_BACKGROUND_ROUTE_STD,
    seed=RUN_SEED,
)
validation_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=VALIDATION_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=VALIDATION_BACKGROUND_ROUTE_MEAN,
    background_route_std=VALIDATION_BACKGROUND_ROUTE_STD,
    seed=RUN_SEED,
)

screening_rng = np.random.default_rng(RUN_SEED)
validation_rng = np.random.default_rng(RUN_SEED + 1)

def route_key(route) -> str:
    return json.dumps([int(node_id) for node_id in route.path_node_ids])

def load_existing_library(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_json(path, lines=True)
    except ValueError:
        return pd.DataFrame()

def risk_adjusted_score(result) -> float:
    return float(-(float(result.average_gtc) + (RISK_AVERSION_K * float(result.std_gtc))))

def coefficient_of_variation(result) -> float:
    mean_gtc = float(result.average_gtc)
    if mean_gtc <= 0:
        return float('inf')
    return float(result.std_gtc / mean_gtc)

def flush_pending_records(library_df: pd.DataFrame, pending_records: list[dict]) -> pd.DataFrame:
    if not pending_records:
        return library_df
    pending_df = pd.DataFrame.from_records(pending_records)
    if library_df.empty:
        merged = pending_df.reset_index(drop=True)
    else:
        merged = pd.concat([library_df, pending_df], ignore_index=True, sort=False)
    pending_records.clear()
    return merged

def serialise_value(value):
    return json.dumps(value, ensure_ascii=False)

def write_library_snapshot(library_df: pd.DataFrame, session_stats: dict[str, int]) -> pd.DataFrame:
    snapshot = library_df.copy()
    if not snapshot.empty and 'route_key' in snapshot.columns:
        snapshot = snapshot.drop_duplicates(subset=['route_key'], keep='first')
        sort_column = 'validation_risk_adjusted_score' if 'validation_risk_adjusted_score' in snapshot.columns else 'screening_risk_adjusted_score'
        snapshot = snapshot.sort_values(sort_column, ascending=False).reset_index(drop=True)

    csv_df = snapshot.copy()
    for column in ['route_node_ids', 'anchor_node_ids', 'ordered_anchor_node_ids', 'path_latlon', 'anchor_latlon']:
        if column in csv_df.columns:
            csv_df[column] = csv_df[column].apply(lambda value: serialise_value(value) if isinstance(value, (list, tuple)) else value)
    csv_df.to_csv(OUTPUT_CSV, index=False)

    best_route = snapshot.iloc[0] if not snapshot.empty else None
    yaml_payload = {
        'experiment': 'B4_good_route_library',
        'higher_is_better': True,
        'source_part2_config': str(PART2_CONFIG),
        'run_seed': RUN_SEED,
        'risk_aversion_k': RISK_AVERSION_K,
        'robustness_cv_ceiling': ROBUSTNESS_CV_CEILING,
        'route_generator': {
            'min_area_m2': int(GENERATOR_SETTINGS.get('min_area_m2', 2_000_000)),
            'anchor_pool_size': int(GENERATOR_SETTINGS.get('anchor_pool_size', 96)),
            'max_attempts': int(GENERATOR_SETTINGS.get('max_attempts', 500)),
            'route_prefix': LIBRARY_ROUTE_PREFIX,
        },
        'screening': {
            'good_percentile': GOOD_PERCENTILE,
            'elite_percentile': ELITE_PERCENTILE,
            'tests': SCREENING_TESTS,
            'background_route_mean': SCREENING_BACKGROUND_ROUTE_MEAN,
            'background_route_std': SCREENING_BACKGROUND_ROUTE_STD,
            'percentile_threshold': SCREENING_SCORE_THRESHOLD,
            'elite_threshold': SCREENING_ELITE_THRESHOLD,
            'mean_plus_2sigma_threshold': SCREENING_SIGMA_THRESHOLD,
            'robustness_ceiling': ROBUSTNESS_CV_CEILING,
        },
        'validation': {
            'tests': VALIDATION_TESTS,
            'background_route_mean': VALIDATION_BACKGROUND_ROUTE_MEAN,
            'background_route_std': VALIDATION_BACKGROUND_ROUTE_STD,
        },
        'library': {
            'jsonl_path': str(OUTPUT_JSONL),
            'csv_path': str(OUTPUT_CSV),
            'initial_route_count': int(INITIAL_ROUTE_COUNT),
            'current_route_count': int(len(snapshot)),
            'library_refresh_every': int(LIBRARY_REFRESH_EVERY),
            'accepted_this_session': int(session_stats.get('accepted_this_session', 0)),
            'attempts_this_session': int(session_stats.get('attempts_this_session', 0)),
            'screened_this_session': int(session_stats.get('screened_this_session', 0)),
            'validated_this_session': int(session_stats.get('validated_this_session', 0)),
            'duplicate_hits_this_session': int(session_stats.get('duplicate_hits_this_session', 0)),
        },
        'summary': {
            'route_count': int(len(snapshot)),
            'mean_validation_average_gtc': float(snapshot['validation_average_gtc'].mean()) if not snapshot.empty and 'validation_average_gtc' in snapshot.columns else None,
            'mean_validation_std_gtc': float(snapshot['validation_std_gtc'].mean()) if not snapshot.empty and 'validation_std_gtc' in snapshot.columns else None,
            'mean_validation_risk_adjusted_score': float(snapshot['validation_risk_adjusted_score'].mean()) if not snapshot.empty and 'validation_risk_adjusted_score' in snapshot.columns else None,
            'std_validation_risk_adjusted_score': float(snapshot['validation_risk_adjusted_score'].std(ddof=0)) if len(snapshot) > 1 and 'validation_risk_adjusted_score' in snapshot.columns else 0.0 if len(snapshot) == 1 and 'validation_risk_adjusted_score' in snapshot.columns else None,
            'best_route_id': str(best_route['route_id']) if best_route is not None and 'route_id' in best_route else None,
            'best_validation_risk_adjusted_score': float(best_route['validation_risk_adjusted_score']) if best_route is not None and 'validation_risk_adjusted_score' in best_route else None,
            'best_validation_average_gtc': float(best_route['validation_average_gtc']) if best_route is not None and 'validation_average_gtc' in best_route else None,
            'best_validation_std_gtc': float(best_route['validation_std_gtc']) if best_route is not None and 'validation_std_gtc' in best_route else None,
        },
    }
    OUTPUT_YAML.write_text(yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=False), encoding='utf-8')
    return snapshot

existing_library = load_existing_library(OUTPUT_JSONL)
library_df = existing_library.copy()
INITIAL_ROUTE_COUNT = int(len(library_df))
seen_keys = set(library_df['route_key'].astype(str)) if not library_df.empty and 'route_key' in library_df.columns else set()
pending_records: list[dict] = []
SESSION_STATS = {
    'attempts_this_session': 0,
    'screened_this_session': 0,
    'validated_this_session': 0,
    'accepted_this_session': 0,
    'duplicate_hits_this_session': 0,
}
write_library_snapshot(library_df, SESSION_STATS)

print(f'Loaded part 2 thresholds from: {PART2_CONFIG}')
print(f'Batch library ready: seed={RUN_SEED}, existing_routes={len(library_df)}')

Loaded part 2 thresholds from: C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\configs\B4_rejection_sampling.yaml
Batch library ready: seed=2410726804, existing_routes=0


### 6.2 Continuous generate-check-append loop
This cell keeps generating routes, screens them with the part 2 thresholds, validates only the survivors, and appends the accepted ones immediately.


In [5]:
SAMPLE_SIZE = 5000

session_stats = SESSION_STATS
progress_bar = tqdm(total=SAMPLE_SIZE, desc='Generating candidate routes', unit='route', dynamic_ncols=True)

evaluate_screen = screening_evaluator.evaluate
evaluate_validate = validation_evaluator.evaluate
risk_score = risk_adjusted_score
cov = coefficient_of_variation
screen_threshold = SCREENING_SCORE_THRESHOLD
robustness_limit = ROBUSTNESS_CV_CEILING
flush_every = LIBRARY_REFRESH_EVERY
append_pending = pending_records.append
seen_add = seen_keys.add

try:
    for candidate_index in range(1, SAMPLE_SIZE + 1):
        route_seed = int(screening_rng.integers(0, np.iinfo(np.int32).max))
        while True:
            try:
                route = sampler.generate_route(route_id=f'{LIBRARY_ROUTE_PREFIX}{candidate_index:06d}', seed=route_seed)
                break
            except ValueError:
                route_seed = int(screening_rng.integers(0, np.iinfo(np.int32).max))

        key = route_key(route)
        session_stats['attempts_this_session'] += 1

        if key in seen_keys:
            session_stats['duplicate_hits_this_session'] += 1
            progress_bar.update(1)
            continue

        screening_result = evaluate_screen(route, n_tests=SCREENING_TESTS, seed=route_seed)
        session_stats['screened_this_session'] += 1
        screening_score = risk_score(screening_result)
        screening_cv = cov(screening_result)
        if screening_score < screen_threshold or screening_cv > robustness_limit:
            progress_bar.update(1)
            continue

        validation_seed = int(validation_rng.integers(0, np.iinfo(np.int32).max))
        validation_result = evaluate_validate(route, n_tests=VALIDATION_TESTS, seed=validation_seed)
        session_stats['validated_this_session'] += 1
        validation_score = risk_score(validation_result)
        validation_cv = cov(validation_result)
        if validation_score < screen_threshold or validation_cv > robustness_limit:
            progress_bar.update(1)
            continue

        record = {
            'route_key': key,
            'route_id': route.route_id,
            'run_seed': RUN_SEED,
            'route_seed': route_seed,
            'validation_seed': validation_seed,
            'route_node_ids': [int(node_id) for node_id in route.path_node_ids],
            'anchor_node_ids': [int(node_id) for node_id in route.anchor_node_ids],
            'ordered_anchor_node_ids': [int(node_id) for node_id in route.ordered_anchor_node_ids],
            'path_latlon': [[float(lat), float(lon)] for lat, lon in route.path_latlon],
            'anchor_latlon': [[float(lat), float(lon)] for lat, lon in route.anchor_latlon],
            'polygon_area_m2': float(route.polygon_area_m2),
            'path_length_m': float(route.path_length_m),
            'attempt_index': int(route.attempt_index),
            'screening_average_gtc': float(screening_result.average_gtc),
            'screening_std_gtc': float(screening_result.std_gtc),
            'screening_risk_adjusted_score': float(screening_score),
            'screening_reward': float(screening_result.reward),
            'screening_average_passenger_gtc_std': float(screening_result.average_passenger_gtc_std),
            'screening_n_tests': int(screening_result.n_tests),
            'screening_coefficient_of_variation': float(screening_cv),
            'validation_average_gtc': float(validation_result.average_gtc),
            'validation_std_gtc': float(validation_result.std_gtc),
            'validation_risk_adjusted_score': float(validation_score),
            'validation_reward': float(validation_result.reward),
            'validation_average_passenger_gtc_std': float(validation_result.average_passenger_gtc_std),
            'validation_n_tests': int(validation_result.n_tests),
            'validation_coefficient_of_variation': float(validation_cv),
        }

        with OUTPUT_JSONL.open('a', encoding='utf-8') as fh:
            fh.write(json.dumps(record, ensure_ascii=False) + '\n')
            fh.flush()

        seen_add(key)
        append_pending(record)
        session_stats['accepted_this_session'] += 1

        if len(pending_records) >= flush_every:
            library_df = flush_pending_records(library_df, pending_records)
            library_df = write_library_snapshot(library_df, session_stats)

        progress_bar.update(1)

except KeyboardInterrupt:
    print('Stopped by user; rerun this cell to resume from the JSONL library.')
finally:
    progress_bar.close()
    library_df = flush_pending_records(library_df, pending_records)
    library_df = write_library_snapshot(library_df, session_stats)
    print(f'Routes in library: {len(library_df)}')
    print(f"Accepted this session: {session_stats['accepted_this_session']}")

Generating candidate routes:   0%|          | 0/5000 [00:00<?, ?route/s]

Stopped by user; rerun this cell to resume from the JSONL library.
Routes in library: 0
Accepted this session: 0


### 6.3 Refresh the saved manifest
Run this cell after stopping the loop if you want to rebuild the CSV and YAML snapshot directly from the JSONL library file.


In [ ]:
if 'validation_scores' not in globals():
    raise RuntimeError('Run the batch screening cells first so validation_scores is available.')

refresh_stats = {
    'accepted_this_session': int(len(validation_scores)),
    'screened_this_session': int(len(screening_scores)) if 'screening_scores' in globals() else int(SAMPLE_SIZE),
    'validated_this_session': int(len(validation_scores)),
    'duplicate_hits_this_session': 0,
    'attempts_this_session': int(SAMPLE_SIZE),
}
library_df = validation_scores.copy().reset_index(drop=True)
library_df = write_library_snapshot(library_df, refresh_stats)

print(f'Refreshed batch library from: {SAMPLE_SIZE:,} sampled routes')
print(f'Routes in library: {len(library_df)}')
display(library_df.head(10))

: 

: 

### 6.4 Inspect the current library
Use this cell to reload the persisted routes and inspect the top of the library after pausing the loop.


In [ ]:
current_library = load_existing_library(OUTPUT_JSONL)
current_library = write_library_snapshot(current_library, SESSION_STATS.copy())
current_manifest = yaml.safe_load(OUTPUT_YAML.read_text(encoding='utf-8')) or {}

print(f'Refreshed library from: {OUTPUT_JSONL}')
print(f'Current library count: {len(current_library)}')
print(f'YAML manifest: {OUTPUT_YAML}')
print(current_manifest.get('summary', {}))
display(current_library.head(10))